# Tutorial 06 Transfer Learning: From Frozen Features to Fine-Tuning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JimengShi/SysEng_Cornell/blob/main/Tutorial_06_Transfer_Learning/tutorial_06_pytorch.ipynb)

Adapt a pretrained MobileNetV2 to five flower classes. Train a new head, continue from that checkpoint with a small part of the backbone unfrozen, and compare both stages using held-out data.

**Start:** Open in Colab → connect → **Runtime → Run all**, or use Shift+Enter to work cell by cell. Required package setup is included below. No local installation, API key, or Drive mount is needed.

**Prerequisites:** Python, a PyTorch training loop, and CNNs (Tutorials 03–05 are helpful).

**Resources:** CPU supported in quick mode; GPU optional. The flower archive is about 218 MB and MobileNetV2 weights about 14 MB. No paid service is required. Allow 60–90 minutes for reading and exercises; downloads and training depend on the runtime.

Each notebook is independent: earlier lessons are helpful but their runtime state and files are not required. While the GitHub repository is private, authorize Colab to access it, or download this notebook and use **File → Upload notebook** in Colab.

## Learning goals and route

By the end you should be able to distinguish pretrained inference, frozen feature extraction, and fine-tuning; inspect trainable parameters; manage BatchNorm buffers; preserve a trained head when unfreezing layers; and choose a checkpoint without using test data.

Unlike Tutorial 05, which concentrates on visual explanations, this lesson studies **what changes during training**. We use the flower example from the upstream lesson as one consistent experiment instead of mixing cats/dogs batches and flower labels.

| Stage | Backbone | New five-class head | Input |
|---|---|---|---|
| ImageNet demonstration | Pretrained, unchanged | Original 1,000-class head | One training image |
| Frozen features | Parameters and BatchNorm buffers fixed | Trainable | Cached feature vectors |
| Fine-tuning | Last two feature blocks trainable; BatchNorm statistics fixed | Continue the learned head | Images, never stale cached features |

Quick mode uses **200 / 50 / 50** training/validation/test images. Larger mode uses an approximately 80/10/10 stratified split of all available images. Both modes download the full archive. We report compute time, not energy or carbon emissions: a faster adaptation run alone does not establish a lower life-cycle footprint, especially when pretraining is excluded.

## 1. Environment

This cell installs missing packages inside the current runtime. It keeps an existing PyTorch/torchvision pair together and does not downgrade Colab’s installed stack. If imports fail after other experiments, start a fresh runtime and run from the top.

In [ ]:
import importlib.util
import subprocess
import sys

packages = {'torch': 'torch', 'torchvision': 'torchvision', 'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'PIL': 'pillow'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if "torch" in missing or "torchvision" in missing:
    missing = sorted(set(missing + ["torch", "torchvision"]))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("Required packages are already installed.")

In [ ]:
import json, platform, random, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset, Subset
import torchvision
from torchvision import datasets, transforms, models
from PIL import Image
from IPython.display import display
%matplotlib inline

SEED = 42
QUICK_RUN = True
DEVICE_PREFERENCE = "auto"  # "cpu" forces CPU; "auto" uses CUDA when available.
assert DEVICE_PREFERENCE in {"auto", "cpu"}
device = torch.device("cuda" if DEVICE_PREFERENCE == "auto" and torch.cuda.is_available() else "cpu")
def seed_everything():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_everything()
torch.set_num_threads(min(2, torch.get_num_threads()))
DATA_DIR = Path("data") / "tutorial_06"
OUTPUT_DIR = Path("outputs") / "tutorial_06"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
versions = {"python": platform.python_version(), "torch": str(torch.__version__),
            "torchvision": str(torchvision.__version__), "numpy": np.__version__,
            "pandas": pd.__version__, "matplotlib": matplotlib.__version__,
            "device": str(device), "seed": SEED, "quick_run": QUICK_RUN}
display(pd.Series(versions, name="Environment"))
run_started = time.perf_counter()

import copy
import hashlib
import math
import shutil
import tarfile
import urllib.request
BATCH_SIZE = 16
HEAD_EPOCHS = 20 if QUICK_RUN else 40
FINE_TUNE_EPOCHS = 2 if QUICK_RUN else 5

## 2. Residual connections, batches, and losses

A residual block computes `y = x + F(x)` when dimensions match; a projection is needed when they do not. Its input derivative contains an identity term, giving gradients a direct path. This can help optimization but does not guarantee non-vanishing gradients or improved test accuracy. MobileNetV2 uses residual connections only in compatible blocks.

The toy calculation below sets the residual branch to zero, so the output and gradient have an exact reference. It is not a trained ResNet.

One optimizer **step** processes a mini-batch. With the usual non-dropping loader, an **epoch** has `ceil(N / batch_size)` steps; the final batch can be smaller.

In [ ]:
x_demo = torch.tensor([[1., 2., 3.]], requires_grad=True)
zero_residual = F.linear(x_demo, torch.zeros(3, 3))
y_demo = x_demo + zero_residual
y_demo.sum().backward()
torch.testing.assert_close(y_demo, x_demo)
torch.testing.assert_close(x_demo.grad, torch.ones_like(x_demo))
print("Zero residual: output", y_demo.detach().tolist(), "input gradient", x_demo.grad.tolist())
print("200 training samples / batch size 16 →", math.ceil(200 / BATCH_SIZE), "steps per epoch")

## 3. Download images, then split by class

The Google-hosted `flower_photos` archive contains daisies, dandelions, roses, sunflowers, and tulips. We retain its `LICENSE.txt`, which contains image attribution. The code's MIT license does not replace image terms.

Downloads are cached and use a temporary file before completion. A SHA-256 digest records the retrieved archive; it is an observed fingerprint, not a separately authenticated checksum. Tar extraction rejects links and unsafe paths. No personal images or credentials are needed.

In [ ]:
DATA_URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
archive_path = DATA_DIR / "flower_photos.tgz"
if not archive_path.exists():
    partial = archive_path.with_suffix(".part")
    print("Downloading flower archive (about 218 MB)...", flush=True)
    with urllib.request.urlopen(DATA_URL, timeout=120) as response, partial.open("wb") as destination:
        shutil.copyfileobj(response, destination)
    partial.replace(archive_path)
archive_sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()
with tarfile.open(archive_path, "r:gz") as archive:
    root = DATA_DIR.resolve()
    for member in archive.getmembers():
        if member.issym() or member.islnk() or not (root / member.name).resolve().is_relative_to(root):
            raise ValueError("Unsafe archive member")
    archive.extractall(root, filter="data")
dataset_root = DATA_DIR / "flower_photos"
license_path = dataset_root / "LICENSE.txt"
assert license_path.exists(), "Dataset attribution file is missing"
class_names = sorted(path.name for path in dataset_root.iterdir() if path.is_dir())
assert class_names == ["daisy", "dandelion", "roses", "sunflowers", "tulips"]
print("Classes:", class_names)
print("Attribution file:", license_path)
print("Archive SHA-256:", archive_sha256)

In [ ]:
train_files, val_files, test_files, split_rows = [], [], [], []
rng = np.random.default_rng(SEED)
for label, name in enumerate(class_names):
    files = sorted((dataset_root / name).glob("*.jpg"))
    order = rng.permutation(len(files))
    if QUICK_RUN:
        counts = [40, 10, 10]
    else:
        counts = [int(0.8*len(files)), int(0.1*len(files))]
        counts.append(len(files)-sum(counts))
    assert sum(counts) <= len(files)
    offset = 0
    for group, count in zip([train_files, val_files, test_files], counts):
        group.extend((files[i], label) for i in order[offset:offset+count])
        offset += count
    split_rows.append({"class":name, "available":len(files), "train":counts[0],
                       "validation":counts[1], "test":counts[2]})
path_sets = [{path.resolve() for path, _ in group} for group in [train_files,val_files,test_files]]
assert not path_sets[0]&path_sets[1] and not path_sets[0]&path_sets[2] and not path_sets[1]&path_sets[2]
assert sum(map(len,path_sets)) == len(set.union(*path_sets))
display(pd.DataFrame(split_rows))
print("Split before preprocessing/loaders:", *map(len,[train_files,val_files,test_files]))

## 4. Match input preprocessing to pretrained weights

The weights enum supplies the resize, crop, normalization, and ImageNet class ordering associated with this model. We use deterministic preprocessing for **both training stages** to keep the comparison focused on continued optimization and unfreezing. Random augmentation is a useful separate experiment, not silently applied only to one stage.

The original ImageNet head has 1,000 outputs; these are not our five flower labels. We first inspect it on a training image, then replace it with a new head. Labels must come from the same weights metadata—an external label file with a background row can introduce an off-by-one error.

In [ ]:
weights = models.MobileNet_V2_Weights.IMAGENET1K_V2
preprocess = weights.transforms()
pretrained = models.mobilenet_v2(weights=weights).to(device).eval()

class ImageFiles(Dataset):
    def __init__(self, files): self.files = files
    def __len__(self): return len(self.files)
    def __getitem__(self, index):
        path, label = self.files[index]
        with Image.open(path) as image:
            return preprocess(image.convert("RGB")), label

def make_loader(dataset, shuffle=False, batch_size=BATCH_SIZE):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0,
                      generator=torch.Generator().manual_seed(SEED))

image_train, image_val, image_test = [ImageFiles(files) for files in [train_files,val_files,test_files]]
mean_rgb = torch.tensor(preprocess.mean).view(3,1,1)
std_rgb = torch.tensor(preprocess.std).view(3,1,1)
def rgb_image(image):
    return (image.detach().cpu()*std_rgb+mean_rgb).clamp(0,1).permute(1,2,0).numpy()
fig, axes = plt.subplots(1,5,figsize=(12,3),layout="constrained")
for label, ax in enumerate(axes):
    index = next(i for i, (_,target) in enumerate(train_files) if target == label)
    image, _ = image_train[index]
    ax.imshow(rgb_image(image)); ax.set_title(class_names[label]); ax.axis("off")
fig.savefig(OUTPUT_DIR/"flower_classes.png",dpi=130)
plt.show()
print(preprocess)

In [ ]:
example_image, example_label = image_train[0]
with torch.no_grad():
    imagenet_logits = pretrained(example_image.unsqueeze(0).to(device))
    top_scores, top_indices = imagenet_logits.softmax(1)[0].topk(5)
display(pd.DataFrame({"ImageNet class":[weights.meta["categories"][i] for i in top_indices.tolist()],
                      "softmax score":top_scores.cpu().tolist()}))
print("Our dataset label:", class_names[example_label])
print("ImageNet head output shape:", tuple(imagenet_logits.shape))
print("Different label spaces: do not score these 1,000-way outputs as five-way flower predictions.")

### Binary versus multiclass prediction

For five mutually exclusive classes, use five raw logits, integer labels, and `CrossEntropyLoss`; `argmax` picks a class. Do not apply softmax before that loss.

For a single binary logit, use `BCEWithLogitsLoss` and a zero-logit threshold (equivalently sigmoid ≥ 0.5). `argmax` along a single output column always returns zero. The following small check makes this common mistake visible; the flower experiment remains five-class throughout.

In [ ]:
binary_logits = torch.tensor([[-2.0], [0.0], [2.0]])
correct_binary = (binary_logits[:,0] >= 0).long()
wrong_binary = binary_logits.argmax(dim=1)
assert correct_binary.tolist() == [0,1,1] and wrong_binary.tolist() == [0,0,0]
print("Threshold predictions:", correct_binary.tolist(), "Single-column argmax:", wrong_binary.tolist())

## 5. Build a genuinely frozen feature extractor

Set `requires_grad` on **parameters**, not a new attribute on a container. Freezing gradients alone does not freeze BatchNorm running statistics: they are buffers, updated in training mode. Our model keeps the entire backbone in evaluation mode during both stages. Fine-tuned convolution weights still receive gradients in evaluation mode; BatchNorm statistics stay fixed for this small dataset.

We keep global-average-pooled 1,280-dimensional features and add a five-class linear head. Inspect actual trainable counts instead of relying on approximate architecture claims.

In [ ]:
class FlowerClassifier(nn.Module):
    def __init__(self, features, classes):
        super().__init__()
        self.features = features
        self.head = nn.Linear(1280, classes)
    def forward(self, images):
        return self.head(self.features(images).mean(dim=(2,3)))
    def train(self, mode=True):
        super().train(mode)
        self.features.eval()  # Keep BatchNorm buffers fixed, including in trainable blocks.
        return self

seed_everything()
flower_model = FlowerClassifier(copy.deepcopy(pretrained.features), len(class_names)).to(device)
flower_model.features.requires_grad_(False)
flower_model.eval()
def parameter_counts(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total":total, "trainable":trainable, "frozen":total-trainable}
head_counts = parameter_counts(flower_model)
assert head_counts["trainable"] == (1280+1)*len(class_names)
print("Frozen-feature stage:", head_counts)
frozen_reference = {k:v.detach().cpu().clone() for k,v in flower_model.features.state_dict().items()}

### Cache once, then fit only the head

Feature caching is valid because the backbone and preprocessing remain fixed. We cache training and validation features; the test set is left untouched until both strategies are fixed. Training reads shuffled cached vectors, while validation selects the lowest cross-entropy checkpoint. Losses are weighted by sample count, so a small last batch is not over-weighted.

In [ ]:
@torch.no_grad()
def cache_features(dataset):
    flower_model.eval()
    features, targets = [], []
    for images, labels in make_loader(dataset):
        features.append(flower_model.features(images.to(device)).mean(dim=(2,3)).cpu())
        targets.append(labels)
    return TensorDataset(torch.cat(features), torch.cat(targets))
cache_started = time.perf_counter()
feature_train, feature_val = cache_features(image_train), cache_features(image_val)
cache_seconds = time.perf_counter()-cache_started
assert feature_train.tensors[0].shape == (len(image_train),1280)
print("Cached training features:", feature_train.tensors[0].shape)

def one_epoch(model, loader, optimizer=None):
    model.train(optimizer is not None)
    loss_sum, correct, count = 0., 0, 0
    with torch.set_grad_enabled(optimizer is not None):
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = model(inputs)
            loss = F.cross_entropy(logits,labels)
            assert torch.isfinite(loss)
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            loss_sum += loss.item()*len(labels)
            correct += (logits.argmax(1)==labels).sum().item()
            count += len(labels)
    return {"loss":loss_sum/count,"accuracy":correct/count}

def fit(model, train_loader, val_loader, optimizer, epochs):
    initial = one_epoch(model,val_loader)
    best_loss, best_epoch = initial["loss"], 0
    best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    history = []
    for epoch in range(1,epochs+1):
        train = one_epoch(model,train_loader,optimizer)
        val = one_epoch(model,val_loader)
        history.append({"epoch":epoch,"train_loss":train["loss"],"val_loss":val["loss"],
                        "train_accuracy":train["accuracy"],"val_accuracy":val["accuracy"]})
        if val["loss"] < best_loss:
            best_loss,best_epoch = val["loss"],epoch
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        print(f"Epoch {epoch}: train {train['accuracy']:.1%}, validation {val['accuracy']:.1%}")
    return pd.DataFrame(history),best_state,best_epoch

head_optimizer = torch.optim.AdamW(flower_model.head.parameters(),lr=0.003,weight_decay=0.001)
head_started = time.perf_counter()
head_history, head_state, head_best_epoch = fit(flower_model.head,
    make_loader(feature_train,shuffle=True),make_loader(feature_val),head_optimizer,HEAD_EPOCHS)
head_seconds = time.perf_counter()-head_started
flower_model.head.load_state_dict(head_state)
flower_model.eval()
assert all(torch.equal(v.cpu(),frozen_reference[k]) for k,v in flower_model.features.state_dict().items())
with torch.no_grad():
    direct = flower_model(image_train[0][0].unsqueeze(0).to(device)).cpu()
    cached = flower_model.head(feature_train.tensors[0][:1].to(device)).cpu()
torch.testing.assert_close(direct,cached,rtol=1e-4,atol=1e-4)
print("Backbone parameters AND buffers unchanged; cached/direct predictions agree.")
print("Best head epoch:",head_best_epoch)

## 6. Continue training: unfreeze explicit blocks, keep the head

Copy the **validation-selected** stage-one model so the frozen strategy remains available for comparison. Preserve its learned head; replacing it here would restart classification training. Unfreeze `features[-2:]` (the final two feature blocks), rather than counting arbitrary parameter tensors.

Use a new optimizer with a smaller backbone learning rate. Previously cached vectors are now stale: train and validate on images. Earlier feature blocks and every BatchNorm running buffer must remain unchanged. We verify that at least one unfrozen weight actually changes before checkpoint selection.

Epoch zero is a valid fine-tuning checkpoint. If no later epoch improves validation loss, retain the starting model. Fine-tuning is not guaranteed to help.

In [ ]:
fine_model = copy.deepcopy(flower_model).to(device)
fine_model.features.requires_grad_(False)
fine_model.features[-2:].requires_grad_(True)
fine_counts = parameter_counts(fine_model)
print("Fine-tuning stage:", fine_counts)
assert fine_counts["trainable"] > head_counts["trainable"]
assert all(torch.equal(v,flower_model.head.state_dict()[k]) for k,v in fine_model.head.state_dict().items())
frozen_names = {name for name,p in fine_model.named_parameters() if not p.requires_grad}
# Buffers (BatchNorm running statistics and counters) are also frozen by train().
frozen_names.update(name for name,_ in fine_model.named_buffers())
protected_before = {k:v.detach().cpu().clone() for k,v in fine_model.state_dict().items() if k in frozen_names}
tail_before = {k:v.detach().cpu().clone() for k,v in fine_model.features[-2:].state_dict().items()}
optimizer = torch.optim.AdamW([
    {"params":fine_model.features[-2:].parameters(),"lr":1e-5},
    {"params":fine_model.head.parameters(),"lr":1e-4}],weight_decay=0.001)
assert {id(p) for group in optimizer.param_groups for p in group["params"]} == {
    id(p) for p in fine_model.parameters() if p.requires_grad}
fine_started = time.perf_counter()
fine_history,fine_state,fine_best_epoch = fit(fine_model,make_loader(image_train,shuffle=True),
    make_loader(image_val),optimizer,FINE_TUNE_EPOCHS)
fine_seconds = time.perf_counter()-fine_started
assert all(torch.equal(fine_model.state_dict()[k].cpu(),v) for k,v in protected_before.items())
tail_changed = any(not torch.equal(v.cpu(),tail_before[k]) for k,v in fine_model.features[-2:].state_dict().items())
assert tail_changed, "No unfrozen weight changed during optimization"
fine_model.load_state_dict(fine_state)
fine_model.eval()
print("Frozen parameters/BatchNorm buffers unchanged; unfrozen weights changed: passed.")
print("Best fine-tuning epoch (0 means keep stage one):",fine_best_epoch)

## 7. Compare learning curves and select using validation

These are sequential stages, not independent equal-budget experiments. Fine-tuning starts from the learned head and receives extra optimization. The plot shows loss/accuracy on the same split, with a boundary between stages. Parameter count is not a complete measure of runtime: even a frozen backbone must compute features.

Record caching and training time separately. They are wall-clock observations for this runtime, not energy measurements. The reported test set must not be used to choose a learning rate, epoch, or strategy.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4),layout="constrained")
for history,offset in [(head_history,0),(fine_history,HEAD_EPOCHS)]:
    for prefix,label,color in [("train","Training","tab:blue"),("val","Validation","tab:orange")]:
        legend = label if offset==0 else None
        axes[0].plot(history.epoch+offset,history[prefix+"_loss"],color=color,label=legend)
        axes[1].plot(history.epoch+offset,history[prefix+"_accuracy"],color=color,label=legend)
for ax in axes:
    ax.axvline(HEAD_EPOCHS+0.5,color="black",ls="--",label="Start fine-tuning")
    ax.set_xlabel("Epoch (frozen head → fine-tuning)");ax.legend(fontsize=8)
axes[0].set_ylabel("Cross-entropy");axes[1].set_ylabel("Accuracy");axes[1].set_ylim(0,1)
fig.savefig(OUTPUT_DIR/"learning_curves.png",dpi=140)
plt.show()
strategies = {"Frozen features":flower_model,"Fine-tuned":fine_model}
validation = {name:one_epoch(model,make_loader(image_val)) for name,model in strategies.items()}
selected = min(validation,key=lambda name:validation[name]["loss"])
display(pd.DataFrame(validation).T)
print("Selected using validation loss:",selected)
print("Caching seconds:",round(cache_seconds,2),"Head fit:",round(head_seconds,2),
      "Fine-tuning fit:",round(fine_seconds,2))

In [ ]:
@torch.no_grad()
def collect_logits(model, dataset):
    model.eval()
    scores, targets = [], []
    for images,labels in make_loader(dataset):
        scores.append(model(images.to(device)).cpu()); targets.append(labels)
    return torch.cat(scores),torch.cat(targets)

test_results, test_logits = {}, {}
for name,model in strategies.items():
    logits,test_targets = collect_logits(model,image_test)
    test_logits[name] = logits
    test_results[name] = {"loss":F.cross_entropy(logits,test_targets).item(),
                         "accuracy":(logits.argmax(1)==test_targets).float().mean().item()}
majority = torch.bincount(torch.tensor([label for _,label in train_files])).argmax()
baseline_accuracy = (test_targets==majority).float().mean().item()
display(pd.DataFrame(test_results).T)
print("Training-majority test baseline:",baseline_accuracy)
predicted = test_logits[selected].argmax(1)
confusion = torch.bincount(len(class_names)*test_targets+predicted,minlength=len(class_names)**2).reshape(5,5)
assert confusion.sum()==len(image_test)
fig,axes = plt.subplots(1,2,figsize=(12,4),layout="constrained")
axes[0].bar(list(test_results),[row["accuracy"] for row in test_results.values()],color=["tab:blue","tab:orange"])
axes[0].axhline(baseline_accuracy,color="black",ls="--",label="Training-majority baseline")
axes[0].set(ylabel="Test accuracy",ylim=(0,1),title="Prespecified strategies (not selected on test)")
axes[0].legend(fontsize=8)
image_plot=axes[1].imshow(confusion.numpy(),cmap="Blues")
axes[1].set(xticks=range(5),yticks=range(5),xticklabels=class_names,yticklabels=class_names,
            xlabel="Predicted",ylabel="True",title=f"Validation-selected: {selected}")
axes[1].tick_params(axis="x",labelrotation=35)
for row in range(5):
    for col in range(5):
        axes[1].text(col,row,str(confusion[row,col].item()),ha="center",va="center",
                     color="white" if confusion[row,col]>confusion.max()/2 else "black")
fig.colorbar(image_plot,ax=axes[1],label="Image count",shrink=0.8)
fig.savefig(OUTPUT_DIR/"test_comparison.png",dpi=140)
plt.show()
# Interleave classes so the first ten shown are not all from the first class.
indices = [i for offset in range(2) for label in range(5)
           for i in [int((test_targets==label).nonzero().flatten()[offset])]]
fig,axes = plt.subplots(2,5,figsize=(12,6),layout="constrained")
for ax,index in zip(axes.flat,indices):
    image,label = image_test[index]; pred=predicted[index].item()
    ax.imshow(rgb_image(image));ax.axis("off")
    ax.set_title(f"True: {class_names[label]}\nPred: {class_names[pred]}",fontsize=9,
                 color="green" if label==pred else "darkred")
fig.savefig(OUTPUT_DIR/"predictions.png",dpi=140)
plt.show()

## 8. Save what inference actually needs

Save the selected weights, class ordering, preprocessing, and model identifier together. Rebuild the architecture with `weights=None`, reload on CPU, and compare logits. A fine-tuned checkpoint already includes the adapted backbone; no second pretrained download is needed for reload.

The quick test set has only 50 images, and class accuracy changes in large increments. We have not audited overlap with ImageNet pretraining. These results teach a workflow; they do not establish deployment readiness or clean independence from pretraining data.

In [ ]:
checkpoint_path = OUTPUT_DIR/"flower_classifier.pt"
chosen = strategies[selected]
torch.save({"state_dict":{k:v.detach().cpu() for k,v in chosen.state_dict().items()},
    "class_names":class_names,"architecture":"mobilenet_v2_features_plus_linear",
    "pretrained_weights":"MobileNet_V2_Weights.IMAGENET1K_V2","selected_strategy":selected,
    "preprocessing":{"crop_size":preprocess.crop_size,"resize_size":preprocess.resize_size,
                     "mean":preprocess.mean,"std":preprocess.std,"interpolation":"bilinear","antialias":True},
    "seed":SEED},checkpoint_path)
saved = torch.load(checkpoint_path,map_location="cpu",weights_only=True)
restored = FlowerClassifier(models.mobilenet_v2(weights=None).features,len(saved["class_names"])).eval()
restored.load_state_dict(saved["state_dict"])
with torch.no_grad():
    restored_logits = restored(image_test[0][0].unsqueeze(0))
torch.testing.assert_close(restored_logits,test_logits[selected][:1],rtol=1e-4,atol=1e-4)
print("CPU checkpoint reload matches original logits.")
results = {"environment":versions,"data_url":DATA_URL,"archive_sha256":archive_sha256,
    "class_names":class_names,"splits":[len(image_train),len(image_val),len(image_test)],
    "class_counts":split_rows,"parameters":{"frozen_stage":head_counts,"fine_tune_stage":fine_counts},
    "epochs":{"head":HEAD_EPOCHS,"fine_tune":FINE_TUNE_EPOCHS},
    "best_epochs":{"head":head_best_epoch,"fine_tune":fine_best_epoch},"validation":validation,
    "test":test_results,"selected_by_validation":selected,"baseline_accuracy":baseline_accuracy,
    "checks":{"head_preserved":True,"frozen_backbone_unchanged":True,"protected_fine_tune_state_unchanged":True,
              "unfrozen_weights_changed":tail_changed,"cached_direct_equivalence":True,"checkpoint_roundtrip":True},
    "timing_seconds":{"cache":cache_seconds,"head_fit":head_seconds,"fine_tune_fit":fine_seconds,
                      "total_after_imports":time.perf_counter()-run_started}}
(OUTPUT_DIR/"metrics.json").write_text(json.dumps(results,indent=2,allow_nan=False))
print("TUTORIAL_06_COMPLETE")
print(json.dumps(results,indent=2))

## 9. Exercises and interpretation

1. Draw which weights and buffers can change in each stage. Why are `requires_grad=False` and `eval()` different?
2. Explain why cached features must be recomputed when backbone weights or preprocessing change.
3. Try unfreezing only the final block versus the final two, selecting on validation. Leave test results out of tuning.
4. Add training-only augmentation as a separately labeled experiment. Use deterministic transforms for validation/test and avoid reusing an incompatible cache.
5. Why can fine-tuning return epoch zero? Does the lowest training loss identify the best model?
6. Increase training-set size with `QUICK_RUN=False`. The held-out sets also change, so quick/full scores are not a controlled estimate of dataset-size effects.
7. Derive the last batch size for 200 samples with batch size 16. Why should epoch loss be sample-weighted?
8. Compare training from scratch as a new experiment, including its compute budget. Do not infer energy/carbon savings from training time alone.

**Concept checks:** Freeze parameters explicitly; keep chosen BatchNorm buffers stable; preserve the learned head; rebuild the optimizer after unfreezing; a single binary logit needs a threshold; ImageNet output indices need ImageNet labels.

## Troubleshooting and next steps

- Run cells from the top after a runtime restart; a saved notebook does not preserve Python variables.
- If a data download fails, check connectivity and rerun that cell. If imports fail, use a fresh runtime.
- Download files from `outputs/tutorial_06/` using Colab’s Files sidebar before disconnecting.
- Change one experimental setting at a time and record what you changed.

## Attribution

License information is available in the repository `LICENSE`.

Sources: [PyTorch transfer learning](https://github.com/pytorch/tutorials/blob/main/beginner_source/transfer_learning_tutorial.py), [MobileNetV2 weights and preprocessing](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.mobilenet_v2.html), [residual learning paper](https://arxiv.org/abs/1512.03385), and [Google flower image dataset workflow](https://www.tensorflow.org/tutorials/load_data/images). The latter is a data-source reference; no TensorFlow code or package is used. Image attributions remain in the downloaded `flower_photos/LICENSE.txt`; weights and images are not redistributed in this repository.

Compared with the upstream lesson, this version removes the inaccessible cats/dogs download, fixes parameter freezing and binary prediction logic, preserves the head across fine-tuning, uses a single consistent label space, and adds held-out evaluation and checkpoint checks.